# Fine-tune YOLOv8n ở 320 px (cho Pi 4)

Model hiện tại train ở **imgsz=640** nhưng Pi chạy ở **320**: người ở xa nhỏ đi một nửa so với lúc train,
recall lớp person chỉ 0.63. Notebook này train tiếp từ `best.pt` đã có, ở đúng 320 px, với augmentation
phóng to/thu nhỏ mạnh hơn cho người ở xa, rồi so sánh hai model **ở 320 px** trên cùng tập val.

Đọc dataset và `best.pt` từ `MyDrive/Colab/YOLOv8n_VEDC` (shortcut tới thư mục của Thịnh, chỉ đọc); ghi kết quả vào thư mục riêng `MyDrive/Colab/YOLOv8n_VEDC_finetune_320`. Chạy lần lượt từ trên xuống. Hết hạn mức giữa chừng: mở lại notebook (tài khoản khác cũng được),
chạy lại ô 1 và ô 3, quá trình train tự tiếp tục từ `last.pt` trên Drive.

Không export TFLite ở đây: bản `int8` của Ultralytics làm điểm số bão hoà ở 0.50-0.61. Gửi `best_320.pt`
và `best_320.onnx` (ô 5) để xuất bằng `YOLO_Movenet/tools/export_tflite.py` trong repo.

## 1. Drive, dữ liệu, thư viện

In [ ]:
!pip install --upgrade ultralytics "pillow<11.0.0" pyyaml -q

import os, shutil, time, yaml, torch
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/Colab/YOLOv8n_VEDC"
drive_dataset_dir = f"{ROOT}/datasets/unified_dataset"
local_dataset_dir = "/content/unified_dataset"
local_project_dir = "/content/runs"
base_weights = f"{ROOT}/runs/yolov8n_person_boat/weights/best.pt"      # model 640 hiện tại
run_name = "yolov8n_person_boat_320"
# Kết quả lưu vào thư mục RIÊNG trong Drive của tài khoản đang chạy Colab, ngoài thư mục YOLOv8n_VEDC của Thịnh
# (thư mục đó chỉ dùng để đọc dataset và best.pt).
OUT_ROOT = "/content/drive/MyDrive/Colab/YOLOv8n_VEDC_finetune_320"
drive_backup_dir = f"{OUT_ROOT}/weights"
SAVE_EVERY = 5                                                           # lưu lên Drive sau mỗi 5 epoch

if not os.path.exists(local_dataset_dir):
    print("[INFO] Copying dataset to local Colab SSD...")
    t = time.time()
    shutil.copytree(drive_dataset_dir, local_dataset_dir)
    print(f"[SUCCESS] Dataset copied in {time.time() - t:.1f} s")

local_data_yaml = os.path.join(local_dataset_dir, "data.yaml")
cfg = yaml.safe_load(open(local_data_yaml))
cfg["path"] = local_dataset_dir
yaml.dump(cfg, open(local_data_yaml, "w"))

assert os.path.exists(base_weights), f"Không thấy {base_weights}"
os.makedirs(drive_backup_dir, exist_ok=True)
print("[INFO] Kết quả sẽ lưu vào:", OUT_ROOT)
print("[INFO] GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

## 2. Điểm xuất phát: model 640 hiện tại, chấm ở 320 px

In [ ]:
from ultralytics import YOLO

def evaluate(weights, label):
    m = YOLO(weights).val(data=local_data_yaml, imgsz=320, batch=64, split="val", plots=False, verbose=False)
    ap50 = m.box.ap50                      # theo lớp: 0 person, 1 boat
    p, r = m.box.p, m.box.r
    row = {"model": label, "mAP50": m.box.map50, "mAP50-95": m.box.map,
           "person mAP50": ap50[0], "person recall": r[0], "person precision": p[0], "boat mAP50": ap50[1]}
    print(" | ".join(f"{k} {v:.3f}" if isinstance(v, float) else f"{v}" for k, v in row.items()))
    return row

before = evaluate(base_weights, "640-trained @320")

## 3. Fine-tune ở 320 px

- train tiếp từ `best.pt` (không phải từ `yolov8n.pt`), learning rate nhỏ để không quên cái đã học;
- `scale=0.7`: ảnh bị thu nhỏ tới 30 % kích thước, tạo thêm người nhỏ như khi ở xa;
- `close_mosaic=5`: 5 epoch cuối tắt mosaic để model quen ảnh nguyên khung như camera;
- lưu `last.pt`, `best.pt`, `results.csv` lên Drive sau mỗi 5 epoch và khi train xong (thư mục `Colab/YOLOv8n_VEDC_finetune_320`); hết hạn mức thì chạy lại ô 1 và ô này để train tiếp từ lần lưu gần nhất (mất tối đa 4 epoch).

In [ ]:
def save_to_drive(trainer, tag):
    try:
        os.makedirs(drive_backup_dir, exist_ok=True)
        src = os.path.join(trainer.save_dir, "weights")
        for name in ("last.pt", "best.pt"):
            if os.path.exists(os.path.join(src, name)):
                shutil.copy(os.path.join(src, name), os.path.join(drive_backup_dir, name))
        csv = os.path.join(trainer.save_dir, "results.csv")
        if os.path.exists(csv):
            shutil.copy(csv, os.path.join(OUT_ROOT, "results.csv"))
        print(f" -> [DRIVE] {tag}: last.pt, best.pt, results.csv saved to {OUT_ROOT}")
    except Exception as e:
        print(f" -> [DRIVE WARNING] {e}")

def every_n_epochs(trainer):
    if (trainer.epoch + 1) % SAVE_EVERY == 0:
        save_to_drive(trainer, f"epoch {trainer.epoch + 1}")

def at_end(trainer):
    save_to_drive(trainer, "training finished")

resume_from = os.path.join(drive_backup_dir, "last.pt")
if os.path.exists(resume_from):
    print(f"[INFO] Resuming from {resume_from}")
    model = YOLO(resume_from)
    model.add_callback("on_fit_epoch_end", every_n_epochs)   # sau validation, best.pt đã cập nhật
    model.add_callback("on_train_end", at_end)
    model.train(resume=True)
else:
    model = YOLO(base_weights)
    model.add_callback("on_fit_epoch_end", every_n_epochs)   # sau validation, best.pt đã cập nhật
    model.add_callback("on_train_end", at_end)
    model.train(
        data=local_data_yaml,
        epochs=30,
        imgsz=320,              # đúng kích thước Pi chạy
        batch=64,
        workers=2,
        device=0,
        cache=True,
        project=local_project_dir,
        name=run_name,
        exist_ok=True,
        optimizer="AdamW",
        lr0=0.001,              # nhỏ: fine-tune, không train lại từ đầu
        cos_lr=True,
        warmup_epochs=1,
        scale=0.7,              # thêm người nhỏ / ở xa
        close_mosaic=5,
        patience=10,
        plots=False,
        save=True,
    )
print("[SUCCESS] Training finished, weights in", drive_backup_dir)

## 4. So sánh trước / sau, cùng tập val, cùng 320 px

In [ ]:
after = evaluate(os.path.join(drive_backup_dir, "best.pt"), "fine-tuned @320")
print()
for key in ("mAP50", "mAP50-95", "person mAP50", "person recall", "person precision", "boat mAP50"):
    print(f"{key:<18} {before[key]:.3f} -> {after[key]:.3f}  ({after[key] - before[key]:+.3f})")

## 5. Xuất ONNX 320 và lưu lên Drive

Gửi lại hai file này (`best_320.pt`, `best_320.onnx` trong `Colab/YOLOv8n_VEDC_finetune_320/weights` trên Drive).
TFLite sẽ được xuất bằng `YOLO_Movenet/tools/export_tflite.py` (toạ độ chuẩn hoá + dynamic range).

In [ ]:
best = os.path.join(drive_backup_dir, "best.pt")
onnx_path = YOLO(best).export(format="onnx", imgsz=320, batch=1, simplify=True, dynamic=False)
shutil.copy(best, os.path.join(drive_backup_dir, "best_320.pt"))
shutil.copy(onnx_path, os.path.join(drive_backup_dir, "best_320.onnx"))
print("[SUCCESS] Saved best_320.pt and best_320.onnx to", drive_backup_dir)